# CivicFix — Image Classifier (EfficientNetB0)

**Before running:**
1. Upload the `dataset/` folder (from `civic-fix/ml/dataset/`) to your Google Drive
2. Make sure Runtime → Change runtime type → **T4 GPU** is selected
3. Update `DATASET_PATH` in Cell 2 to match your Drive path

Expected Drive structure:
```
MyDrive/
  civicfix_dataset/
    train/
      cracks/ garbage/ open_manhole/ pothole/ water_overflow/
    val/
      ...
    test/
      ...
    class_names.json
```

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Config — update DATASET_PATH to match your Drive ────────
import os

# ✏️  Change this to where you uploaded the dataset folder in Drive
DATASET_PATH = '/content/drive/MyDrive/civicfix_dataset'

# Model outputs will be saved here (also in Drive so they persist)
OUTPUT_PATH  = '/content/drive/MyDrive/civicfix_model'

IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
LR_HEAD     = 1e-3
LR_FINE     = 1e-5
SEED        = 42

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Verify paths exist
for split in ['train', 'val', 'test']:
    p = os.path.join(DATASET_PATH, split)
    assert os.path.isdir(p), f'Missing: {p}'
    classes = os.listdir(p)
    counts  = {c: len(os.listdir(os.path.join(p, c))) for c in classes}
    print(f'{split:6s}: {counts}')

print('\nConfig OK.')

In [ ]:
# ── Cell 3: Check GPU ────────────────────────────────────────────────
import tensorflow as tf

print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if not gpus:
    print('⚠️  No GPU detected — go to Runtime > Change runtime type > T4 GPU')
else:
    print('✅ GPU ready')

In [ ]:
# ── Cell 4: Data generators with augmentation ────────────────────────
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    zoom_range=0.20,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)
val_aug = ImageDataGenerator(rescale=1./255)

train_gen = train_aug.flow_from_directory(
    os.path.join(DATASET_PATH, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEED,
)
val_gen = val_aug.flow_from_directory(
    os.path.join(DATASET_PATH, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEED,
)
test_gen = val_aug.flow_from_directory(
    os.path.join(DATASET_PATH, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

NUM_CLASSES = len(train_gen.class_indices)
CLASS_NAMES = {str(v): k for k, v in train_gen.class_indices.items()}
print(f'\nNum classes : {NUM_CLASSES}')
print(f'Class map   : {train_gen.class_indices}')
print(f'Train batches: {len(train_gen)}  |  Val batches: {len(val_gen)}')

In [ ]:
# ── Cell 5: Visualise a batch (sanity check) ─────────────────────────
import matplotlib.pyplot as plt

imgs, labels = next(train_gen)
idx_to_name  = {v: k for k, v in train_gen.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.set_title(idx_to_name[np.argmax(labels[i])], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample training images (after augmentation)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 6: Build EfficientNetB0 model ──────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False   # freeze for head training

inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()
print(f'\nTrainable params (head only): {sum(tf.size(w).numpy() for w in model.trainable_weights):,}')

In [ ]:
# ── Cell 7: Phase 1 — Train classifier head ──────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(LR_HEAD),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

ckpt_path = os.path.join(OUTPUT_PATH, 'best_model.keras')

cb_head = [
    keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, verbose=1,
    ),
]

print('=== Phase 1: Classifier head ===')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_HEAD,
    callbacks=cb_head,
)

In [ ]:
# ── Cell 8: Phase 2 — Fine-tune top layers ───────────────────────────
base_model.trainable = True

# Keep early layers frozen — fine-tune last 30 layers only
for layer in base_model.layers[:-30]:
    layer.trainable = False

fine_tunable = sum(1 for w in model.trainable_weights)
print(f'Trainable weight tensors after unfreeze: {fine_tunable}')

model.compile(
    optimizer=keras.optimizers.Adam(LR_FINE),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

cb_fine = [
    keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=4, verbose=1,
    ),
]

print('=== Phase 2: Fine-tuning ===')
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINE,
    callbacks=cb_fine,
)

In [ ]:
# ── Cell 9: Save final model + class map ────────────────────────────
import json

final_path = os.path.join(OUTPUT_PATH, 'final_model.keras')
model.save(final_path)
print(f'Final model saved → {final_path}')

label_map = {
    'class_names': CLASS_NAMES,          # {"0": "cracks", "1": "garbage", ...}
    'class_indices': train_gen.class_indices,  # {"cracks": 0, ...}
    'num_classes': NUM_CLASSES,
}
meta_path = os.path.join(OUTPUT_PATH, 'class_names.json')
with open(meta_path, 'w') as f:
    json.dump(label_map, f, indent=2)
print(f'Class map saved  → {meta_path}')
print(f'\nAll outputs at: {OUTPUT_PATH}')

In [ ]:
# ── Cell 10: Evaluate on test set ───────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best checkpoint for evaluation
best_model = keras.models.load_model(ckpt_path)

test_gen.reset()
loss, acc = best_model.evaluate(test_gen, verbose=0)
print(f'Test accuracy : {acc:.4f}')
print(f'Test loss     : {loss:.4f}')

test_gen.reset()
preds  = best_model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes
names  = list(test_gen.class_indices.keys())

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=names))

In [ ]:
# ── Cell 11: Confusion matrix ────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=names, yticklabels=names,
)
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_PATH, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=100)
plt.show()
print(f'Saved → {cm_path}')

In [ ]:
# ── Cell 12: Training curves ─────────────────────────────────────────
def merge(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(merge(history1, history2, 'accuracy'),     label='Train')
axes[0].plot(merge(history1, history2, 'val_accuracy'), label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(merge(history1, history2, 'loss'),     label='Train')
axes[1].plot(merge(history1, history2, 'val_loss'), label='Val')
axes[1].set_title('Loss'); axes[1].legend()

plt.tight_layout()
curve_path = os.path.join(OUTPUT_PATH, 'training_history.png')
plt.savefig(curve_path, dpi=100)
plt.show()
print(f'Saved → {curve_path}')

In [ ]:
# ── Cell 13: Quick single-image inference test ───────────────────────
# Drop any image path here to test the model manually
import pathlib
from tensorflow.keras.preprocessing import image as kimage

def predict_image(img_path, mdl=best_model):
    img   = kimage.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr   = kimage.img_to_array(img) / 255.0
    arr   = np.expand_dims(arr, axis=0)
    probs = mdl.predict(arr, verbose=0)[0]
    idx   = np.argmax(probs)
    return {
        'category':   idx_to_name[idx],
        'confidence': float(round(probs[idx], 4)),
        'all_probs':  {idx_to_name[i]: float(round(p, 4)) for i, p in enumerate(probs)},
    }

idx_to_name = {v: k for k, v in train_gen.class_indices.items()}

# Example — replace with any test image path
sample_dir = os.path.join(DATASET_PATH, 'test')
sample_cls  = os.listdir(sample_dir)[0]
sample_file = os.listdir(os.path.join(sample_dir, sample_cls))[0]
sample_path = os.path.join(sample_dir, sample_cls, sample_file)

result = predict_image(sample_path)
print(f'File      : {sample_file}')
print(f'True class: {sample_cls}')
print(f'Predicted : {result["category"]}  ({result["confidence"]*100:.1f}%)')
print(f'All probs : {result["all_probs"]}')

plt.imshow(kimage.load_img(sample_path))
plt.title(f'True: {sample_cls}  |  Pred: {result["category"]} ({result["confidence"]*100:.1f}%)')
plt.axis('off'); plt.show()

## Done!

Download these two files from your Drive (`civicfix_model/`) and bring them back:

| File | Purpose |
|---|---|
| `best_model.keras` | The trained model |
| `class_names.json` | Label index map for inference |

Place both inside `civic-fix/ml/model/` and we'll wire up the Flask inference server next.